# ICNALE GRA — download & preprocess

*Holistic essay score band (Low / Mid / High)*

**What it is.** Asian-learner L2 English essays, each rated on holistic and analytic scales by many trained raters. This is an **automated writing evaluation** task: whole essays, not sentences.

**Difficulty of the labeling judgment:** ★★☆ — moderate, but a different shape of task: long texts and an ordered scale.

**Licence:** ⚠️ **Research use only — NOT redistributable.** Requires registration. Nothing derived from it may be committed to git or included in your submission bundle.  
**Cite:** Ishikawa, S. *The ICNALE Global Rating Archives.*

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

> This notebook is **generated** from `scripts/reshape.py`. The reshaping code below is the same code `scripts/prep_datasets.py` runs — not a copy of it. If you want to change how the data is reshaped, edit `reshape.py` and re-run `scripts/_generate_download_notebooks.py`.

## Step 1 — Get the data (this one is manual)

ICNALE GRA is released for research use behind a registration form that emails you a password. There is nothing to automate, and that is deliberate — the licence does not permit redistribution.

1. Register at <https://language.sakura.ne.jp/icnale/download.html> and wait for the password.
2. Download and unpack `ICNALE_GRA_2.x.zip`.
3. From its rating tables, export a CSV with **exactly two columns**, `text` and `score`, and upload it here (or put it in `data/raw/icnale/essays_scores.csv`).

In Colab, the cell below opens a file picker.

In [ ]:
# In Colab: uncomment to upload your essays_scores.csv
# from google.colab import files; files.upload()

RAW_FILE = "essays_scores.csv"

## Step 2 — Look at the raw format

In [ ]:
import csv

with open(RAW_FILE, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("columns:", reader.fieldnames)
    for row, _ in zip(reader, range(2)):
        print(round(float(row["score"]), 2), "|", row["text"][:100], "...")

## Step 3 — Reshape into the canonical schema

One decision, and it is entirely yours: **where do the band boundaries go?**

The defaults below (`< 4` = Low, `< 7` = Mid, else High) are **placeholders** — round numbers, not the ICNALE rubric. Where you cut decides how hard the task is and how balanced the classes are, so set them from the rubric you are actually using and **say what you chose in your report**.

⚠️ These labels are **ordered** (Low < Mid < High) but they are *not* alphabetical. So pass the order explicitly when you evaluate — `LABELS_ORDER = ["Low", "Mid", "High"]` — or the weighted κ will be computed over `High < Low < Mid`, which means nothing.

In [ ]:
def reid(items):
    """Renumber ids sequentially from 1, keeping the current order."""
    renumbered = []
    next_id = 1
    for item in items:
        new_item = dict(item)
        new_item["id"] = next_id
        renumbered.append(new_item)
        next_id = next_id + 1
    return renumbered

def reshape_icnale(csv_path, low_below=4.0, mid_below=7.0):
    """Band a numeric holistic score into Low / Mid / High.

    THE CUT-OFFS ARE PLACEHOLDERS. 4 and 7 are not from the ICNALE rubric - they are
    round numbers. Where you put the boundaries decides how hard the task is and how
    balanced the classes are, so set them from the rubric you are actually using and
    say what you chose in your report.
    """
    rows = []
    skipped = 0
    with open(csv_path, encoding="utf-8-sig", newline="") as handle:
        for record in csv.DictReader(handle):
            text = (record.get("text") or "").strip()
            raw_score = (record.get("score") or "").strip()
            if not text or not raw_score:
                continue
            try:
                score = float(raw_score)
            except ValueError:
                skipped = skipped + 1      # a non-numeric cell: report it, do not crash
                continue
            if score < low_below:
                label = "Low"
            elif score < mid_below:
                label = "Mid"
            else:
                label = "High"
            rows.append({"id": 0, "text": text, "label": label})
    if skipped:
        print("  note: skipped", skipped, "row(s) whose score cell was not a number.")
    return reid(rows)

In [ ]:
rows = reshape_icnale(RAW_FILE)      # try low_below=..., mid_below=... too

## Step 4 — Check the label balance

In [ ]:
from collections import Counter

print("total items:", len(rows))
print("label counts:", dict(Counter(item["label"] for item in rows)))
rows[:3]        # peek at the first three reshaped items

## A note on what you just built

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is *not* your gold set.

Your gold set comes next, in the project notebook: `sample_pool` draws a *balanced* subset from this pool (equal items per label), which is what makes precision, recall, F1 and the confusion matrix readable. Keeping the two separate also leaves the unsampled items free to serve as few-shot examples without leaking the answers you are testing on.

So: build the pool once, here. Sample from it there.

## Step 5 — Save it

⚠️ Keep this file **out of git** and **out of your submission bundle**. `.gitignore` and `scripts/make_submission.py` both exclude anything with `icnale` in the name — please leave that in place.

In [ ]:
# Save the pool. Two places you might want it:
#   * this repo, if you cloned it:  "../data/pools/icnale_pool.json"
#   * your Google Drive, so it survives the Colab runtime resetting
import json

OUT_FILE = "icnale_pool.json"

# In Colab, uncomment these two lines to write straight to your Drive:
# from google.colab import drive; drive.mount("/content/drive")
# OUT_FILE = "/content/drive/MyDrive/icnale_pool.json"

with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", OUT_FILE)